# Australian Greyhound Racing — Staking & Strategy Backtester

**Prerequisite:** Run `greyhound_eda.ipynb` first to produce `greyhound_races.parquet`.

This notebook backtests five strategies on the full historical dataset:

| # | Strategy | Selection | Staking |
|---|----------|-----------|--------|
| 1 | Lay favourite | BSP favourite every race | Flat $10 liability |
| 2 | Back favourite | BSP favourite every race | Flat $10 stake |
| 3 | Back shortener | Favourite that shortened ≥15% morning→BSP | Flat $10 stake |
| 4 | Back value | Runner where BSP > calibrated fair price | Flat $10 stake |
| 5 | Kelly lay | Lay favourite when implied prob > true prob | Kelly-scaled liability |

All strategies use **BSP execution** (Betfair Starting Price) with **5% commission** on winning bets.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

OUTPUT_DIR  = './greyhound_output'
COMMISSION  = 0.05
FLAT_STAKE  = 10.0   # $ per bet (back bets) or $ liability (lay bets)
KELLY_FRAC  = 0.25   # fractional Kelly to limit variance
STARTING_BANK = 1_000.0

races = pd.read_parquet(os.path.join(OUTPUT_DIR, 'greyhound_races.parquet'))
cal   = pd.read_csv(os.path.join(OUTPUT_DIR, 'bsp_calibration.csv'))

races['event_dt'] = pd.to_datetime(races['event_dt'])
races = races.sort_values('event_dt').reset_index(drop=True)

print(f'Loaded {len(races):,} races  ({races["event_dt"].min().date()} → {races["event_dt"].max().date()})')
print(f'With BSP: {races["has_bsp"].sum():,}  ({races["has_bsp"].mean()*100:.1f}%)')

## Calibration — Build True Probability Model

Fit a simple isotonic regression over the BSP calibration curve to convert any BSP into an empirical true win probability.

In [ ]:
from sklearn.isotonic import IsotonicRegression

# Load runner-level data for calibration fit
runners = pd.read_parquet(os.path.join(OUTPUT_DIR, 'greyhound_runners.parquet'))
runners = runners[runners['BSP'].notna() & (runners['BSP'] > 1.01)].copy()
runners['implied_prob'] = 1.0 / runners['BSP']

iso = IsotonicRegression(out_of_bounds='clip', increasing=True)
iso.fit(runners['implied_prob'], runners['WIN_LOSE'])

def bsp_to_true_prob(bsp_series):
    """Convert BSP to calibrated true win probability via isotonic regression."""
    implied = 1.0 / bsp_series.clip(lower=1.01)
    return pd.Series(iso.predict(implied), index=bsp_series.index)

# Validate
test_bsps = [1.5, 2.0, 3.0, 5.0, 10.0, 20.0]
print('Calibration model — BSP → true probability:')
print(f'{"BSP":>6s}  {"Implied":>8s}  {"True prob":>10s}  {"Edge":>8s}')
for b in test_bsps:
    implied = 1/b
    true_p  = iso.predict([[implied]])[0]
    edge    = true_p - implied
    print(f'{b:>6.1f}  {implied*100:>7.1f}%   {true_p*100:>9.1f}%  {edge*100:>+7.2f}pp')

## Backtesting Framework

In [ ]:
def backtest_back(selections, starting_bank=STARTING_BANK, commission=COMMISSION):
    """
    Backtest a series of back bets at BSP.

    selections: DataFrame with columns:
        event_dt, bsp, won (1/0), stake

    Returns per-bet P&L DataFrame and summary dict.
    """
    sel = selections.copy().reset_index(drop=True)
    sel['pnl'] = np.where(
        sel['won'] == 1,
        sel['stake'] * (sel['bsp'] - 1) * (1 - commission),
        -sel['stake']
    )
    sel['bank'] = starting_bank + sel['pnl'].cumsum()
    sel['drawdown'] = sel['bank'] - sel['bank'].cummax()
    return sel


def backtest_lay(selections, starting_bank=STARTING_BANK, commission=COMMISSION):
    """
    Backtest a series of lay bets at BSP.

    selections: DataFrame with columns:
        event_dt, bsp, won (1/0 — 1 = runner WON = lay LOSES), liability

    Lay mechanics:
        back_stake = liability / (bsp - 1)
        if runner doesn't win: profit = back_stake * (1 - commission)
        if runner wins:        loss   = liability
    """
    sel = selections.copy().reset_index(drop=True)
    back_stake = sel['liability'] / (sel['bsp'] - 1)
    sel['pnl'] = np.where(
        sel['won'] == 0,
        back_stake * (1 - commission),
        -sel['liability']
    )
    sel['bank'] = starting_bank + sel['pnl'].cumsum()
    sel['drawdown'] = sel['bank'] - sel['bank'].cummax()
    return sel


def summary(result, name, stake_col='stake'):
    n       = len(result)
    wins    = result['won'].sum() if 'won' in result.columns else (result['pnl'] > 0).sum()
    total_staked = result[stake_col].sum()
    total_pnl    = result['pnl'].sum()
    roi          = total_pnl / total_staked * 100 if total_staked > 0 else 0
    max_dd       = result['drawdown'].min()
    final_bank   = result['bank'].iloc[-1]
    monthly_pnl  = result.set_index('event_dt')['pnl'].resample('ME').sum()
    sharpe       = monthly_pnl.mean() / monthly_pnl.std() * np.sqrt(12) if monthly_pnl.std() > 0 else 0
    return {
        'Strategy':      name,
        'Bets':          n,
        'Win rate':      f'{wins/n*100:.1f}%',
        'Total P&L':     f'${total_pnl:+,.0f}',
        'ROI':           f'{roi:+.2f}%',
        'Max drawdown':  f'${max_dd:,.0f}',
        'Final bank':    f'${final_bank:,.0f}',
        'Ann. Sharpe':   f'{sharpe:.2f}',
    }

## Strategy 1 — Lay the Favourite (Flat Liability)

In [ ]:
s1 = races[races['fav_bsp'].notna()][['event_dt','fav_bsp','fav_won']].copy()
s1 = s1.rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s1['liability'] = FLAT_STAKE
s1['stake']     = s1['liability'] / (s1['bsp'] - 1)
r1 = backtest_lay(s1)
sum1 = summary(r1, 'Lay Favourite (flat $10 liability)', stake_col='liability')
print(sum1)

## Strategy 2 — Back the Favourite (Flat Stake)

In [ ]:
s2 = races[races['fav_bsp'].notna()][['event_dt','fav_bsp','fav_won']].copy()
s2 = s2.rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s2['stake'] = FLAT_STAKE
r2 = backtest_back(s2)
sum2 = summary(r2, 'Back Favourite (flat $10)', stake_col='stake')
print(sum2)

## Strategy 3 — Back Morning Shorteners

Back the favourite only when its price shortened ≥15% from morning market to BSP (informed money signal).

In [ ]:
s3 = races[races['fav_bsp'].notna() & races['fav_morningwap'].notna()].copy()
s3['drift'] = (s3['fav_morningwap'] - s3['fav_bsp']) / s3['fav_morningwap']
s3 = s3[s3['drift'] >= 0.15][['event_dt','fav_bsp','fav_won']].copy()
s3 = s3.rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s3['stake'] = FLAT_STAKE
r3 = backtest_back(s3)
sum3 = summary(r3, 'Back Shortener ≥15% (flat $10)', stake_col='stake')
print(f'Selections: {len(s3):,}')
print(sum3)

## Strategy 4 — Back Value (Calibrated Edge)

Back the favourite when the calibrated true probability exceeds the BSP implied probability by ≥2pp (positive expected value).

In [ ]:
s4 = races[races['fav_bsp'].notna()].copy()
s4['true_prob']    = bsp_to_true_prob(s4['fav_bsp'])
s4['implied_prob'] = 1.0 / s4['fav_bsp']
s4['edge']         = s4['true_prob'] - s4['implied_prob']
s4 = s4[s4['edge'] >= 0.02][['event_dt','fav_bsp','fav_won','edge']].copy()
s4 = s4.rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s4['stake'] = FLAT_STAKE
r4 = backtest_back(s4)
sum4 = summary(r4, 'Back Value ≥2pp edge (flat $10)', stake_col='stake')
print(f'Selections: {len(s4):,}')
print(sum4)

## Strategy 5 — Kelly Lay (Calibrated)

Lay the favourite when the BSP implied probability exceeds the calibrated true probability (favourite is overbet).
Stake via fractional Kelly on the liability.

In [ ]:
s5 = races[races['fav_bsp'].notna()].copy()
s5['true_prob']    = bsp_to_true_prob(s5['fav_bsp'])
s5['implied_prob'] = 1.0 / s5['fav_bsp']
s5['lay_edge']     = s5['implied_prob'] - s5['true_prob']   # positive = favourite overbet

# Kelly fraction for laying:
# f = (p_lay_win * (bsp-1) - p_lay_lose) / (bsp-1)
# where p_lay_win = 1 - true_prob, p_lay_lose = true_prob
s5 = s5[s5['lay_edge'] > 0].copy()
q = 1 - s5['true_prob']   # prob lay wins
b = s5['fav_bsp'] - 1     # lay odds - 1 (exposure multiple)
s5['kelly_f']  = ((q * b - s5['true_prob']) / b).clip(0, 0.5)
s5['kelly_f'] *= KELLY_FRAC   # fractional Kelly

# Bank-scaled liability (start with STARTING_BANK, update as bank changes)
bank = STARTING_BANK
liabilities = []
for _, row in s5.iterrows():
    liab = bank * row['kelly_f']
    liabilities.append(liab)
    back_stake = liab / (row['fav_bsp'] - 1)
    if row['fav_won'] == 0:
        bank += back_stake * (1 - COMMISSION)
    else:
        bank -= liab
    bank = max(bank, 0)

s5 = s5[['event_dt','fav_bsp','fav_won']].copy()
s5 = s5.rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s5['liability'] = liabilities
s5['stake']     = s5['liability'] / (s5['bsp'] - 1)
r5 = backtest_lay(s5)
sum5 = summary(r5, f'Kelly Lay (frac={KELLY_FRAC})', stake_col='liability')
print(f'Selections: {len(s5):,}')
print(sum5)

## Results Summary

In [ ]:
summaries = [sum1, sum2, sum3, sum4, sum5]
summary_df = pd.DataFrame(summaries)
print('\n=== Strategy Comparison ===')
print(summary_df.to_string(index=False))

## Equity Curves

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))
fig.suptitle('Strategy Backtests — Australian Greyhound BSP', fontsize=13)

results = [
    (r1, 'Lay Favourite',       '#e74c3c'),
    (r2, 'Back Favourite',      '#3498db'),
    (r3, 'Back Shortener ≥15%', '#2ecc71'),
    (r4, 'Back Value ≥2pp',     '#f39c12'),
    (r5, 'Kelly Lay',           '#9b59b6'),
]

ax1 = axes[0]
for r, label, color in results:
    ax1.plot(r['event_dt'], r['bank'], label=label, color=color, linewidth=1.5, alpha=0.85)
ax1.axhline(STARTING_BANK, color='black', linewidth=0.8, linestyle='--')
ax1.set_ylabel('Bank ($)')
ax1.set_title('Bank Value Over Time')
ax1.legend(fontsize=8)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

ax2 = axes[1]
for r, label, color in results:
    ax2.plot(r['event_dt'], r['drawdown'], label=label, color=color, linewidth=1.2, alpha=0.75)
ax2.fill_between(results[0][0]['event_dt'], results[0][0]['drawdown'], 0,
                  alpha=0.05, color=results[0][2])
ax2.set_ylabel('Drawdown ($)')
ax2.set_title('Drawdown Over Time')
ax2.legend(fontsize=8)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

## Monthly P&L Breakdown

In [ ]:
fig, axes = plt.subplots(len(results), 1, figsize=(14, 3*len(results)), sharex=True)
fig.suptitle('Monthly P&L by Strategy', fontsize=13)

for ax, (r, label, color) in zip(axes, results):
    monthly = r.set_index('event_dt')['pnl'].resample('ME').sum()
    bar_colors = ['#2ecc71' if p >= 0 else '#e74c3c' for p in monthly]
    ax.bar(monthly.index, monthly.values, color=bar_colors, alpha=0.85, width=20)
    ax.axhline(0, color='black', linewidth=0.6)
    ax.set_ylabel('P&L ($)')
    ax.set_title(label, fontsize=9)
    pos_months = (monthly > 0).sum()
    ax.text(0.01, 0.95, f'{pos_months}/{len(monthly)} profitable months',
             transform=ax.transAxes, fontsize=7, va='top')

plt.tight_layout()
plt.show()

## Walk-Forward Validation

Train the calibration model on 70% of data, test on the remaining 30%.
Prevents look-ahead bias in the value/Kelly strategies.

In [ ]:
from sklearn.isotonic import IsotonicRegression

# Chronological split
split_date = races['event_dt'].quantile(0.70)
print(f'Train: up to {split_date.date()}   Test: from {split_date.date()}')

train_runners = runners[runners['EVENT_DT'] < split_date]
test_races    = races[races['event_dt'] >= split_date].copy()

# Refit calibration on training data only
iso_wf = IsotonicRegression(out_of_bounds='clip', increasing=True)
tr = train_runners[train_runners['BSP'].notna()].copy()
iso_wf.fit(1.0 / tr['BSP'].clip(lower=1.01), tr['WIN_LOSE'])

def bsp_to_true_prob_wf(bsp_series):
    implied = 1.0 / bsp_series.clip(lower=1.01)
    return pd.Series(iso_wf.predict(implied), index=bsp_series.index)

# Re-run Strategies 3, 4, 5 on test set only
wf_results = {}

# Strategy 3 — shortener
s3t = test_races[test_races['fav_bsp'].notna() & test_races['fav_morningwap'].notna()].copy()
s3t['drift'] = (s3t['fav_morningwap'] - s3t['fav_bsp']) / s3t['fav_morningwap']
s3t = s3t[s3t['drift'] >= 0.15].rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s3t['stake'] = FLAT_STAKE
wf_results['Back Shortener (OOS)'] = backtest_back(s3t)

# Strategy 4 — value
s4t = test_races[test_races['fav_bsp'].notna()].copy()
s4t['true_prob'] = bsp_to_true_prob_wf(s4t['fav_bsp'])
s4t['edge']      = s4t['true_prob'] - 1.0 / s4t['fav_bsp']
s4t = s4t[s4t['edge'] >= 0.02].rename(columns={'fav_bsp':'bsp','fav_won':'won'})
s4t['stake'] = FLAT_STAKE
wf_results['Back Value (OOS)'] = backtest_back(s4t)

print('\nOut-of-sample results:')
for name, res in wf_results.items():
    sc = 'stake' if 'stake' in res.columns else 'liability'
    print(summary(res, name, stake_col=sc))

## ROI by BSP Odds Bucket

In [ ]:
# Show ROI per odds bucket for back-favourite strategy (most data)
roi_df = r2.copy()
roi_df['odds_bucket'] = pd.cut(roi_df['bsp'],
                                bins=[1, 2, 3, 4, 6, 10, 20, 100],
                                labels=['1-2', '2-3', '3-4', '4-6', '6-10', '10-20', '20+'])

bucket_roi = roi_df.groupby('odds_bucket', observed=True).apply(
    lambda d: pd.Series({
        'n':        len(d),
        'win_rate': d['won'].mean() * 100,
        'total_pnl': d['pnl'].sum(),
        'total_staked': d['stake'].sum(),
        'roi': d['pnl'].sum() / d['stake'].sum() * 100
    })
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Back Favourite — ROI by Odds Bucket', fontsize=12)

colors_roi = ['#2ecc71' if r > 0 else '#e74c3c' for r in bucket_roi['roi']]
axes[0].bar(bucket_roi['odds_bucket'], bucket_roi['roi'], color=colors_roi, alpha=0.85)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_xlabel('BSP odds range')
axes[0].set_ylabel('ROI (%)')
axes[0].set_title('ROI by Odds Bucket')

axes[1].bar(bucket_roi['odds_bucket'], bucket_roi['win_rate'], color='steelblue', alpha=0.85)
axes[1].set_xlabel('BSP odds range')
axes[1].set_ylabel('Win rate (%)')
axes[1].set_title('Win Rate by Odds Bucket')

plt.tight_layout()
plt.show()

print(bucket_roi.assign(
    win_rate=lambda d: d['win_rate'].round(1),
    roi=lambda d: d['roi'].round(2),
    total_pnl=lambda d: d['total_pnl'].round(0)
).to_string(index=False))

## Next Steps

Based on the backtest results, the most promising directions are:

1. **Feature engineering** — combine drift + volume + trap + grade into a single selection model (logistic regression or XGBoost)
2. **Venue-specific calibration** — some tracks may have persistent favourite biases
3. **Grade filtering** — if Juvenile/Maiden races show different calibration, build separate models
4. **Staking refinement** — if any strategy shows +EV, switch from flat to Kelly; use 25% fractional Kelly to manage variance
5. **RL agent** — use the real BSP data to train the `BettingEnv` agent on historical greyhound markets rather than synthetic odds

## Strategy 6 — Implied Probability Profiling & Dutching

Analyse each race through the lens of **implied probability structure**:
- Is there a dominant "one-out" favourite or is the race genuinely contested?
- Does dutching the top-N runners by BSP produce positive EV after commission?
- What combined implied probability threshold, if any, creates edge?

In [ ]:
# ── Build per-race implied probability matrices ───────────────────────────────
# Columns r1_bsp → r8_bsp are already sorted by BSP rank (r1 = BSP favourite)

BSP_COLS  = [f'r{i}_bsp' for i in range(1, 9)]
WON_COLS  = [f'r{i}_won' for i in range(1, 9)]

df = races[races['has_bsp'] == 1].copy().reset_index(drop=True)

bsp_mat = df[BSP_COLS].values.astype(float)   # (n_races, 8)
won_mat = df[WON_COLS].values.astype(float)   # (n_races, 8)  0/1/NaN

# Implied probability matrix — 0 for missing runners
imp_mat = np.where(np.isnan(bsp_mat) | (bsp_mat <= 1.01), 0.0, 1.0 / bsp_mat)

# ── Race-level features ───────────────────────────────────────────────────────
df['overround']  = imp_mat.sum(axis=1)                    # market book (>1 = overround)
df['n_bsp']      = (imp_mat > 0).sum(axis=1)              # runners with valid BSP

# Normalised (fair) probability = implied / overround
norm_mat = np.where(df['overround'].values[:, None] > 0,
                    imp_mat / df['overround'].values[:, None], 0.0)

df['p1'] = norm_mat[:, 0]   # favourite normalised prob
df['p2'] = norm_mat[:, 1]   # 2nd favourite
df['p3'] = norm_mat[:, 2]   # 3rd favourite

# Dominance ratio: how much bigger is the favourite vs 2nd?
df['dominance'] = np.where(df['p2'] > 0, df['p1'] / df['p2'], np.nan)

# Herfindahl-Hirschman Index — race concentration (high = one-sided)
df['hhi'] = (norm_mat ** 2).sum(axis=1)

# ── 1. One-out favourite classification ──────────────────────────────────────
DOMINANCE_BINS   = [0, 1.2, 1.5, 2.0, 3.0, 100]
DOMINANCE_LABELS = ['Contested\n(<1.2×)', 'Slight\n(1.2–1.5×)',
                    'Clear\n(1.5–2.0×)', 'Strong\n(2.0–3.0×)', 'One-out\n(>3×)']

df['dom_class'] = pd.cut(df['dominance'], bins=DOMINANCE_BINS, labels=DOMINANCE_LABELS)

dom_stats = df.groupby('dom_class', observed=True).agg(
    n         = ('fav_won', 'count'),
    fav_wr    = ('fav_won', 'mean'),
    med_bsp   = ('fav_bsp', 'median'),
    med_p1    = ('p1',      'median'),
    med_p2    = ('p2',      'median'),
).reset_index()
dom_stats['back_roi'] = (
    dom_stats['fav_wr'] * (dom_stats['med_bsp'] - 1) * (1 - COMMISSION)
    - (1 - dom_stats['fav_wr'])
)

print('=== One-Out Favourite: Race Classification ===')
print(f'{"Class":<18} {"N":>7} {"Fav WR":>8} {"Med BSP":>8} {"Back ROI":>9}')
print('-' * 54)
for _, r in dom_stats.iterrows():
    flag = ' ◄' if r['back_roi'] > 0 else ''
    print(f'{str(r["dom_class"]).replace(chr(10)," "):<18} {int(r["n"]):>7,} '
          f'{r["fav_wr"]*100:>7.1f}% {r["med_bsp"]:>8.2f} '
          f'{r["back_roi"]*100:>+8.2f}%{flag}')

# Distribution of dominance
dom_clean = df['dominance'].replace([np.inf, -np.inf], np.nan).dropna()
print(f'\nDominance ratio — median: {dom_clean.median():.2f}×  '
      f'p75: {dom_clean.quantile(0.75):.2f}×  '
      f'p90: {dom_clean.quantile(0.90):.2f}×  '
      f'p99: {dom_clean.quantile(0.99):.2f}×')
print(f'One-out favourites (>3×): {(dom_clean > 3).mean()*100:.1f}% of races '
      f'({int((dom_clean > 3).sum()):,} races)')

# ── 2. Dutch top-N scan ───────────────────────────────────────────────────────
# For each N: back all top-N runners at BSP so each returns the same profit.
# Dutch stake on runner i = Total_stake × imp_i / sum(imp_1..N)
# Dutch return (if any wins) = Total_stake / sum(imp_1..N)
# Net ROI = hit_rate × (1-commission)/sum_imp_N - 1

print('\n=== Dutch Top-N: Full Dataset ===')
print(f'{"N":>3}  {"Races":>7}  {"Sum imp%":>9}  {"Dutch odds":>10}  '
      f'{"Hit rate":>9}  {"Gross ROI":>10}  {"Net ROI":>9}')
print('-' * 66)

dutch_rows = []
for n in range(1, 9):
    top_n_imp  = imp_mat[:, :n].sum(axis=1)         # raw combined implied prob
    top_n_won  = np.nansum(won_mat[:, :n], axis=1)  # 1 if any top-n runner won
    mask       = (df['n_bsp'].values >= n) & (top_n_imp > 0)

    sum_imp    = top_n_imp[mask].mean()
    dutch_odds = (1.0 / top_n_imp[mask]).mean()
    hit_rate   = top_n_won[mask].mean()

    # Per-race net ROI (commission on profit only when winner is in top-n)
    pnl_vec = np.where(
        top_n_won[mask] == 1,
        (1.0 / top_n_imp[mask] - 1) * (1 - COMMISSION),   # profit leg
        -1.0                                                  # loss leg (normalised to $1 stake)
    )
    net_roi = pnl_vec.mean()
    gross   = hit_rate * dutch_odds - 1

    flag = ' ◄' if net_roi > 0 else ''
    print(f'{n:>3}  {mask.sum():>7,}  {sum_imp*100:>8.1f}%  '
          f'{dutch_odds:>9.3f}×  {hit_rate*100:>8.1f}%  '
          f'{gross*100:>+9.2f}%  {net_roi*100:>+8.2f}%{flag}')
    dutch_rows.append({'n': n, 'mask': mask, 'net_roi': net_roi,
                       'hit_rate': hit_rate, 'top_n_imp': top_n_imp,
                       'top_n_won': top_n_won, 'n_races': mask.sum()})

# ── 3. Dutch top-2 conditioned on dominance class ────────────────────────────
top2_imp = imp_mat[:, :2].sum(axis=1)
top2_won = np.nansum(won_mat[:, :2], axis=1)
pnl2 = np.where(
    top2_won == 1,
    (1.0 / np.where(top2_imp > 0, top2_imp, np.nan) - 1) * (1 - COMMISSION),
    -1.0
)
df['dutch2_pnl'] = pnl2
df['dutch2_hit'] = top2_won
df['top2_imp']   = top2_imp

print('\n=== Dutch Top-2 by Dominance Class ===')
print(f'{"Class":<20} {"N":>7}  {"Hit rate":>9}  {"Sum imp%":>10}  {"Net ROI":>9}')
print('-' * 57)
grp = df[df['n_bsp'] >= 2].groupby('dom_class', observed=True).agg(
    n        = ('dutch2_pnl', 'count'),
    hit_rate = ('dutch2_hit', 'mean'),
    mean_imp = ('top2_imp',   'mean'),
    mean_roi = ('dutch2_pnl', 'mean'),
).reset_index()
for _, r in grp.iterrows():
    flag = ' ◄' if r['mean_roi'] > 0 else ''
    print(f'{str(r["dom_class"]).replace(chr(10)," "):<20} {int(r["n"]):>7,}  '
          f'{r["hit_rate"]*100:>8.1f}%  {r["mean_imp"]*100:>9.1f}%  '
          f'{r["mean_roi"]*100:>+8.2f}%{flag}')

# ── 4. Dutch top-2 by combined implied probability bucket ─────────────────────
df['top2_bucket'] = pd.cut(
    df['top2_imp'],
    bins=[0, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.90, 1.10, 2.0],
    labels=['<55%','55–60%','60–65%','65–70%','70–75%','75–80%','80–90%','90–110%','>110%']
)
imp_grp = df[df['n_bsp'] >= 2].groupby('top2_bucket', observed=True).agg(
    n        = ('dutch2_pnl', 'count'),
    hit_rate = ('dutch2_hit', 'mean'),
    mean_imp = ('top2_imp',   'mean'),
    mean_roi = ('dutch2_pnl', 'mean'),
    med_fav  = ('fav_bsp',    'median'),
).reset_index()

print('\n=== Dutch Top-2 by Combined Implied Probability ===')
print(f'{"Bucket":>9}  {"N":>7}  {"Hit rate":>9}  {"Sum imp%":>10}  {"Net ROI":>9}  {"Med fav":>8}')
print('-' * 60)
for _, r in imp_grp.iterrows():
    flag = ' ◄' if r['mean_roi'] > 0 else ''
    print(f'{str(r["top2_bucket"]):>9}  {int(r["n"]):>7,}  '
          f'{r["hit_rate"]*100:>8.1f}%  {r["mean_imp"]*100:>9.1f}%  '
          f'{r["mean_roi"]*100:>+8.2f}%  {r["med_fav"]:>7.2f}{flag}')

# ── 5. Best dutch strategy: equity simulation ─────────────────────────────────
# Filter: dutch top-2 only in "one-out" races (dominance > 2×) for concrete P&L trace
sel_mask = (df['n_bsp'] >= 2) & (df['dominance'] >= 2.0)
sel_df   = df[sel_mask].copy().reset_index(drop=True)

bank = STARTING_BANK
bank_series = []
for _, row in sel_df.iterrows():
    stake = FLAT_STAKE
    if row['dutch2_hit'] == 1:
        ret   = (1.0 / row['top2_imp']) if row['top2_imp'] > 0 else 0
        profit = (ret - 1) * stake * (1 - COMMISSION)
    else:
        profit = -stake
    bank += profit
    bank_series.append(bank)

sel_df['bank']     = bank_series
sel_df['pnl']      = sel_df['bank'].diff().fillna(sel_df['bank'].iloc[0] - STARTING_BANK)
sel_df['drawdown'] = sel_df['bank'] - sel_df['bank'].cummax()

total_pnl   = sel_df['bank'].iloc[-1] - STARTING_BANK
total_staked = len(sel_df) * FLAT_STAKE
roi          = total_pnl / total_staked * 100
monthly_pnl  = sel_df.set_index('event_dt')['pnl'].resample('ME').sum()
sharpe       = monthly_pnl.mean() / monthly_pnl.std() * np.sqrt(12) if monthly_pnl.std() > 0 else 0

print(f'\n=== Dutch Top-2 (dominance ≥ 2×) — Equity Simulation ===')
print(f'  Bets         : {len(sel_df):,}')
print(f'  Hit rate     : {sel_df["dutch2_hit"].mean()*100:.1f}%')
print(f'  Total P&L    : ${total_pnl:+,.0f}')
print(f'  ROI          : {roi:+.2f}%')
print(f'  Max drawdown : ${sel_df["drawdown"].min():,.0f}')
print(f'  Final bank   : ${sel_df["bank"].iloc[-1]:,.0f}')
print(f'  Ann. Sharpe  : {sharpe:.2f}')

# ── 6. Visualisations ────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Implied Probability & Dutching Analysis', fontsize=13)

# Panel 1: Fav win rate + back ROI by dominance class
ax = axes[0, 0]
dom_clean_labels = [str(l).replace('\n', ' ') for l in dom_stats['dom_class']]
x = np.arange(len(dom_stats))
bars = ax.bar(x, dom_stats['fav_wr'] * 100,
              color=plt.cm.RdYlGn(np.linspace(0.25, 0.85, len(dom_stats))), alpha=0.85)
ax.axhline(df['fav_won'].mean() * 100, color='black', linewidth=1,
           linestyle='--', label='Overall avg')
ax2t = ax.twinx()
ax2t.plot(x, dom_stats['back_roi'] * 100, 'ko-', linewidth=1.5, markersize=5, label='Back ROI')
ax2t.axhline(0, color='red', linewidth=0.8, linestyle=':')
ax2t.set_ylabel('Back ROI (%)', color='black', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(dom_clean_labels, fontsize=7)
ax.set_ylabel('Favourite win rate (%)')
ax.set_title('Fav Win Rate & Back ROI by Dominance')
ax.legend(fontsize=7, loc='upper left')

# Panel 2: Dutch top-N net ROI
ax2 = axes[0, 1]
dr_ns   = [r['n'] for r in dutch_rows]
dr_rois = [r['net_roi'] * 100 for r in dutch_rows]
dr_hits = [r['hit_rate'] * 100 for r in dutch_rows]
ax2b = ax2.twinx()
ax2.bar(dr_ns, dr_hits, alpha=0.35, color='steelblue', label='Hit rate %')
ax2b.plot(dr_ns, dr_rois, 'rs-', linewidth=2, markersize=6, label='Net ROI %')
ax2b.axhline(0, color='black', linewidth=0.8, linestyle='--')
for n, roi_v in zip(dr_ns, dr_rois):
    ax2b.text(n, roi_v + 0.15, f'{roi_v:+.1f}%', ha='center', fontsize=7,
              color='darkgreen' if roi_v > 0 else 'darkred')
ax2.set_xlabel('Runners dutched (top N by BSP)')
ax2.set_ylabel('Hit rate (%)', color='steelblue')
ax2b.set_ylabel('Net ROI (%)', color='red')
ax2.set_title('Dutch Top-N: Hit Rate vs Net ROI (all races)')
ax2.set_xticks(dr_ns)
lines1, lab1 = ax2.get_legend_handles_labels()
lines2, lab2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1 + lines2, lab1 + lab2, fontsize=7)

# Panel 3: Dutch top-2 ROI by combined implied probability bucket
ax3 = axes[1, 0]
colors3 = ['#2ecc71' if v > 0 else '#e74c3c' for v in imp_grp['mean_roi']]
ax3.bar(range(len(imp_grp)), imp_grp['mean_roi'] * 100, color=colors3, alpha=0.85)
ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_xticks(range(len(imp_grp)))
ax3.set_xticklabels(imp_grp['top2_bucket'], rotation=30, ha='right', fontsize=8)
ax3.set_title('Dutch Top-2 ROI by Combined Implied %')
ax3.set_ylabel('Net ROI (%)')
ax3.yaxis.set_major_formatter(mticker.FormatStrFormatter('%+.1f%%'))
for i, (_, r) in enumerate(imp_grp.iterrows()):
    ax3.text(i, r['mean_roi'] * 100 + (0.05 if r['mean_roi'] >= 0 else -0.2),
             f'n={int(r["n"]):,}', ha='center', fontsize=6)

# Panel 4: Equity curve — Dutch top-2 in strong-favourite races
ax4 = axes[1, 1]
ax4.plot(sel_df['event_dt'], sel_df['bank'], color='darkorange', linewidth=1.5)
ax4.axhline(STARTING_BANK, color='black', linewidth=0.8, linestyle='--', label='Starting bank')
ax4.fill_between(sel_df['event_dt'], sel_df['drawdown'] + sel_df['bank'],
                  sel_df['bank'], alpha=0.15, color='red', label='Drawdown')
ax4.set_title(f'Dutch Top-2 (dominance ≥ 2×): Equity Curve\n'
              f'ROI={roi:+.2f}%  Sharpe={sharpe:.2f}')
ax4.set_ylabel('Bank ($)')
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax4.legend(fontsize=8)

plt.tight_layout()
plt.show()


## Strategy 7 — Odds × Market Rank & Hypothesis Testing

Five targeted analyses:
1. **Odds × rank matrix** — does a $3.00 second favourite win more than a $2.80 favourite?
2. **Implied probability × favouritism** — does rank carry signal beyond price alone?
3. **H1: Trap × Distance × Dominance** — back one-out favourites at long distances
4. **H2: Morning drift × Dominance** — back shortening one-out favourites
5. **H3: Lay the upset** — lay the 'favourite' in genuinely contested races

In [ ]:
# ── Build long-format runner table from r1–r8 columns ────────────────────────
# Each row = one runner with their BSP, rank within race, and outcome

BSP_COLS  = [f'r{i}_bsp'  for i in range(1, 9)]
WON_COLS  = [f'r{i}_won'  for i in range(1, 9)]
TRAP_COLS = [f'r{i}_trap' for i in range(1, 9)]

_df = races[races['has_bsp'] == 1].copy().reset_index(drop=True)

# Compute derived race fields needed for hypotheses
bsp_mat = _df[BSP_COLS].values.astype(float)
imp_mat = np.where(np.isnan(bsp_mat) | (bsp_mat <= 1.01), 0.0, 1.0 / bsp_mat)
_df['overround']  = imp_mat.sum(axis=1)
_df['n_bsp']      = (imp_mat > 0).sum(axis=1)
norm_mat = np.where(_df['overround'].values[:, None] > 0,
                    imp_mat / _df['overround'].values[:, None], 0.0)
_df['dominance']  = np.where(norm_mat[:, 1] > 0, norm_mat[:, 0] / norm_mat[:, 1], np.nan)
_df['fav_drift']  = np.where(
    _df['fav_morningwap'].notna() & _df['fav_bsp'].notna(),
    (_df['fav_morningwap'] - _df['fav_bsp']) / _df['fav_morningwap'],
    np.nan
)

# Build runner-level long table
chunks = []
for rank in range(1, 9):
    tmp = _df[['event_dt', 'dominance', 'distance', 'fav_drift', 'n_bsp',
               'fav_bsp', 'fav_won',
               f'r{rank}_bsp', f'r{rank}_won', f'r{rank}_trap']].copy()
    tmp = tmp[tmp[f'r{rank}_bsp'].notna() & (tmp[f'r{rank}_bsp'] > 1.01)].copy()
    tmp.rename(columns={
        f'r{rank}_bsp':  'bsp',
        f'r{rank}_won':  'won',
        f'r{rank}_trap': 'trap',
    }, inplace=True)
    tmp['rank'] = rank
    chunks.append(tmp)

rdf = pd.concat(chunks, ignore_index=True)
rdf['implied_prob'] = 1.0 / rdf['bsp']

ODDS_BINS   = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 7.0, 10.0, 20.0, 100.0]
ODDS_LABELS = ['<1.5','1.5-2','2-2.5','2.5-3','3-4','4-5','5-7','7-10','10-20','20+']
rdf['odds_bucket'] = pd.cut(rdf['bsp'], bins=ODDS_BINS, labels=ODDS_LABELS)

print(f'Runner-level table: {len(rdf):,} rows  (ranks 1–8 with valid BSP)')

# ── Analysis 1: Win rate by odds bucket × market rank ────────────────────────
pivot_wr = rdf.groupby(['odds_bucket', 'rank'], observed=True)['won'].agg(
    actual_wr='mean', n='count'
).reset_index()
pivot_wr['implied_wr'] = 1.0 / pd.cut(
    pivot_wr['odds_bucket'].map(dict(zip(ODDS_LABELS,
        [1.25,1.75,2.25,2.75,3.5,4.5,6.0,8.5,15.0,40.0]))),
    bins=[-1,100], labels=False
)
# Use midpoint BSP per bucket for implied
midpoints = dict(zip(ODDS_LABELS, [1.25,1.75,2.25,2.75,3.5,4.5,6.0,8.5,15.0,40.0]))
pivot_wr['mid_bsp']     = pivot_wr['odds_bucket'].map(midpoints)
pivot_wr['implied_wr']  = 1.0 / pivot_wr['mid_bsp']
pivot_wr['edge_pp']     = (pivot_wr['actual_wr'] - pivot_wr['implied_wr']) * 100
pivot_wr['back_roi_pct']= (
    pivot_wr['actual_wr'] * (pivot_wr['mid_bsp'] - 1) * (1 - COMMISSION)
    - (1 - pivot_wr['actual_wr'])
) * 100

# Print matrix: edge (pp) by odds bucket × rank
print('\n=== Edge (actual − implied, pp) by Odds Bucket × Market Rank ===')
SHOW_RANKS = [1, 2, 3, 4]
header = f'{"Bucket":<10}' + ''.join(f'  Rank{r:>1}' for r in SHOW_RANKS)
print(header)
print('-' * (10 + 8 * len(SHOW_RANKS)))
for bucket in ODDS_LABELS:
    row_str = f'{bucket:<10}'
    for r in SHOW_RANKS:
        sub = pivot_wr[(pivot_wr['odds_bucket'] == bucket) & (pivot_wr['rank'] == r)]
        if sub.empty or sub['n'].iloc[0] < 200:
            row_str += '      -'
        else:
            e = sub['edge_pp'].iloc[0]
            row_str += f'  {e:>+5.2f}'
    print(row_str)

# ── Analysis 2: Same odds, different rank — head-to-head ─────────────────────
# The core question: at the SAME BSP, does rank matter?
# Compare rank-1 vs rank-2 at overlapping odds ranges

print('\n=== Same Odds, Different Rank: Win Rate Comparison ===')
print(f'{"Odds bucket":<12} {"Rank1 WR":>9} {"Rank2 WR":>9} {"Rank3 WR":>9} '
      f'{"Rank1 N":>8} {"Rank2 N":>8}  {"2nd>1st?":>9}')
print('-' * 72)
for bucket in ODDS_LABELS:
    sub1 = pivot_wr[(pivot_wr['odds_bucket'] == bucket) & (pivot_wr['rank'] == 1)]
    sub2 = pivot_wr[(pivot_wr['odds_bucket'] == bucket) & (pivot_wr['rank'] == 2)]
    sub3 = pivot_wr[(pivot_wr['odds_bucket'] == bucket) & (pivot_wr['rank'] == 3)]
    if sub1.empty or sub2.empty or sub1['n'].iloc[0] < 500:
        continue
    wr1 = sub1['actual_wr'].iloc[0]
    wr2 = sub2['actual_wr'].iloc[0] if not sub2.empty else float('nan')
    wr3 = sub3['actual_wr'].iloc[0] if not sub3.empty else float('nan')
    n1  = int(sub1['n'].iloc[0])
    n2  = int(sub2['n'].iloc[0]) if not sub2.empty else 0
    flag = ' ◄ YES' if wr2 > wr1 else ''
    print(f'{bucket:<12} {wr1*100:>8.2f}% {wr2*100:>8.2f}% {wr3*100:>8.2f}% '
          f'{n1:>8,} {n2:>8,}{flag}')

# ── Analysis 3: Back ROI by rank and odds ─────────────────────────────────────
print('\n=== Back ROI by Rank (post-commission, $10 flat) ===')
rank_stats = rdf.groupby('rank', observed=True).apply(lambda d: pd.Series({
    'n':         len(d),
    'win_rate':  d['won'].mean(),
    'med_bsp':   d['bsp'].median(),
    'mean_bsp':  d['bsp'].mean(),
    'back_roi':  (d['won'] * (d['bsp'] - 1) * (1 - COMMISSION) - (1 - d['won'])).mean(),
})).reset_index()
print(f'{"Rank":>5} {"N":>8} {"Win rate":>9} {"Med BSP":>8} {"Back ROI":>9}')
print('-' * 44)
for _, r in rank_stats.iterrows():
    flag = ' ◄' if r['back_roi'] > 0 else ''
    print(f'{int(r["rank"]):>5} {int(r["n"]):>8,} {r["win_rate"]*100:>8.2f}% '
          f'{r["med_bsp"]:>8.2f} {r["back_roi"]*100:>+8.2f}%{flag}')

# ── Hypothesis 1: Trap × Distance × Dominance ────────────────────────────────
print('\n═══ H1: Trap × Distance × Dominance ═══')
print('Filter: distance > 560m  AND  dominance > 3×  AND  trap 1-4\n')

h1 = _df[
    (_df['distance'] > 560) &
    (_df['dominance'] > 3.0) &
    (_df['fav_bsp'].notna())
].copy()
h1['fav_trap_int'] = h1['fav_trap'].fillna(-1).astype(int)

# Full filtered set
h1_sel = h1.rename(columns={'fav_bsp': 'bsp', 'fav_won': 'won'}).copy()
h1_sel['stake'] = FLAT_STAKE
r_h1 = backtest_back(h1_sel)
s_h1 = summary(r_h1, 'H1: Long dist + one-out (all traps)', stake_col='stake')
print(f'  All traps:')
print(f'  Bets={s_h1["Bets"]}  WR={s_h1["Win rate"]}  ROI={s_h1["ROI"]}  Sharpe={s_h1["Ann. Sharpe"]}')

# Trap 1-4 only
h1t = h1[h1['fav_trap_int'].between(1, 4)].rename(
    columns={'fav_bsp': 'bsp', 'fav_won': 'won'}).copy()
h1t['stake'] = FLAT_STAKE
r_h1t = backtest_back(h1t)
s_h1t = summary(r_h1t, 'H1: Long dist + one-out + trap 1-4', stake_col='stake')
print(f'  Trap 1-4 only:')
print(f'  Bets={s_h1t["Bets"]}  WR={s_h1t["Win rate"]}  ROI={s_h1t["ROI"]}  Sharpe={s_h1t["Ann. Sharpe"]}')

# By trap
print('\n  Win rate by trap (long dist, one-out favourites):')
trap_grp = h1.groupby('fav_trap_int')['fav_won'].agg(['mean','count']).reset_index()
trap_grp.columns = ['trap','win_rate','n']
trap_grp = trap_grp[trap_grp['trap'].between(1, 8) & (trap_grp['n'] >= 30)]
for _, r in trap_grp.iterrows():
    bar = '█' * int(r['win_rate'] * 50)
    print(f'    Trap {int(r["trap"])}: {r["win_rate"]*100:5.1f}%  n={int(r["n"]):,}  {bar}')

# ── Hypothesis 2: Morning drift × Dominance ──────────────────────────────────
print('\n═══ H2: Morning Drift × One-Out Dominance ═══')
print('Filter: dominance > 3×  AND  morning drift ≥ 10%\n')

DRIFT_THRESHOLDS = [0.05, 0.10, 0.15, 0.20]
print(f'  {"Drift thresh":>14} {"Bets":>7} {"Win rate":>9} {"ROI":>9} {"Sharpe":>8}')
print('  ' + '-' * 52)
for thr in DRIFT_THRESHOLDS:
    h2 = _df[
        (_df['dominance'] > 3.0) &
        (_df['fav_drift'] >= thr) &
        (_df['fav_bsp'].notna())
    ].rename(columns={'fav_bsp': 'bsp', 'fav_won': 'won'}).copy()
    if len(h2) < 50:
        print(f'  {thr*100:>13.0f}%  {"<50 bets — skip":>45}')
        continue
    h2['stake'] = FLAT_STAKE
    r_h2 = backtest_back(h2)
    s_h2 = summary(r_h2, '', stake_col='stake')
    flag = ' ◄' if float(s_h2['ROI'].replace('%','').replace('+','')) > 0 else ''
    print(f'  {thr*100:>13.0f}%  {s_h2["Bets"]:>7}  {s_h2["Win rate"]:>9}  '
          f'{s_h2["ROI"]:>9}  {s_h2["Ann. Sharpe"]:>7}{flag}')

# Also test drift × dominance × distance
h2_long = _df[
    (_df['dominance'] > 3.0) &
    (_df['fav_drift'] >= 0.10) &
    (_df['distance'] > 450) &
    (_df['fav_bsp'].notna())
].rename(columns={'fav_bsp': 'bsp', 'fav_won': 'won'}).copy()
if len(h2_long) >= 50:
    h2_long['stake'] = FLAT_STAKE
    r_h2l = backtest_back(h2_long)
    s_h2l = summary(r_h2l, '', stake_col='stake')
    flag = ' ◄' if float(s_h2l['ROI'].replace('%','').replace('+','')) > 0 else ''
    print(f'\n  With distance >450m filter:')
    print(f'  Bets={s_h2l["Bets"]}  WR={s_h2l["Win rate"]}  ROI={s_h2l["ROI"]}  '
          f'Sharpe={s_h2l["Ann. Sharpe"]}{flag}')

# ── Hypothesis 3: Lay the upset (contested races) ────────────────────────────
print('\n═══ H3: Lay the "Favourite" in Contested Races ═══')
print('Filter: dominance < 1.2×  (favourite barely distinguishable from 2nd)\n')

CONTEST_THRESHOLDS = [1.05, 1.10, 1.15, 1.20]
print(f'  {"Dom thresh":>12} {"Bets":>7} {"Fav WR":>8} {"ROI":>9} {"Sharpe":>8}')
print('  ' + '-' * 50)
for thr in CONTEST_THRESHOLDS:
    h3 = _df[
        (_df['dominance'] < thr) &
        (_df['fav_bsp'].notna())
    ].rename(columns={'fav_bsp': 'bsp', 'fav_won': 'won'}).copy()
    if len(h3) < 100:
        continue
    h3['liability'] = FLAT_STAKE
    h3['stake']     = h3['liability'] / (h3['bsp'] - 1)
    r_h3 = backtest_lay(h3)
    s_h3 = summary(r_h3, '', stake_col='liability')
    flag = ' ◄' if float(s_h3['ROI'].replace('%','').replace('+','')) > 0 else ''
    print(f'  {thr:>12.2f}×  {s_h3["Bets"]:>7}  {s_h3["Win rate"]:>7}  '
          f'{s_h3["ROI"]:>9}  {s_h3["Ann. Sharpe"]:>7}{flag}')

# ── Visualisations ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Odds × Rank Analysis & Hypothesis Tests', fontsize=13)

# Panel 1: Win rate heatmap — odds bucket × rank
ax = axes[0, 0]
heat_data = pivot_wr[pivot_wr['rank'].isin([1,2,3,4])].pivot(
    index='rank', columns='odds_bucket', values='edge_pp'
)
# Drop columns with no data
heat_data = heat_data.dropna(axis=1, how='all').fillna(0)
im = ax.imshow(heat_data.values, aspect='auto', cmap='RdYlGn',
                vmin=-3, vmax=3)
ax.set_xticks(range(len(heat_data.columns)))
ax.set_xticklabels(heat_data.columns, rotation=30, ha='right', fontsize=7)
ax.set_yticks(range(len(heat_data.index)))
ax.set_yticklabels([f'Rank {r}' for r in heat_data.index])
ax.set_title('Edge (pp) = Actual − Implied WR\nby Odds Bucket × Market Rank')
plt.colorbar(im, ax=ax, label='Edge (pp)')
for i in range(len(heat_data.index)):
    for j in range(len(heat_data.columns)):
        v = heat_data.values[i, j]
        ax.text(j, i, f'{v:+.1f}', ha='center', va='center', fontsize=7,
                color='white' if abs(v) > 1.5 else 'black')

# Panel 2: Win rate by rank at key odds buckets
ax2 = axes[0, 1]
key_buckets = ['2-2.5', '2.5-3', '3-4', '4-5', '5-7']
rank_colors = ['#e74c3c','#3498db','#2ecc71','#f39c12']
for ri, (rank, color) in enumerate(zip([1,2,3,4], rank_colors)):
    wrs, buckets_plot = [], []
    for b in key_buckets:
        sub = pivot_wr[(pivot_wr['odds_bucket'] == b) & (pivot_wr['rank'] == rank)]
        if not sub.empty and sub['n'].iloc[0] >= 200:
            wrs.append(sub['actual_wr'].iloc[0] * 100)
            buckets_plot.append(b)
    if wrs:
        x_pos = [key_buckets.index(b) + ri * 0.18 - 0.27 for b in buckets_plot]
        ax2.bar(x_pos, wrs, width=0.17, color=color, alpha=0.85, label=f'Rank {rank}')
ax2.set_xticks(range(len(key_buckets)))
ax2.set_xticklabels(key_buckets, fontsize=8)
ax2.set_ylabel('Win rate (%)')
ax2.set_title('Win Rate at Same Odds by Market Rank\n(Does rank matter?)')
ax2.legend(fontsize=7)

# Panel 3: Back ROI by rank
ax3 = axes[0, 2]
rs_plot = rank_stats[rank_stats['n'] > 1000]
colors_r = ['#2ecc71' if r > 0 else '#e74c3c' for r in rs_plot['back_roi']]
ax3.bar(rs_plot['rank'].astype(int), rs_plot['back_roi'] * 100, color=colors_r, alpha=0.85)
ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_xlabel('Market rank (1 = favourite)')
ax3.set_ylabel('Back ROI (%)')
ax3.set_title('Back ROI by Market Rank\n(flat $10, all odds)')
ax3.yaxis.set_major_formatter(mticker.FormatStrFormatter('%+.1f%%'))

# Panel 4: H1 equity — trap × distance × dominance
ax4 = axes[1, 0]
ax4.plot(r_h1['event_dt'],  r_h1['bank'],  color='steelblue', label='All traps', linewidth=1.5)
ax4.plot(r_h1t['event_dt'], r_h1t['bank'], color='darkorange', label='Trap 1-4', linewidth=1.5)
ax4.axhline(STARTING_BANK, color='black', linewidth=0.8, linestyle='--')
ax4.set_title('H1: Long Dist + One-Out Favourite')
ax4.set_ylabel('Bank ($)')
ax4.legend(fontsize=8)
ax4.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Panel 5: H2 equity — drift × dominance
ax5 = axes[1, 1]
h2_best = _df[
    (_df['dominance'] > 3.0) & (_df['fav_drift'] >= 0.10) & (_df['fav_bsp'].notna())
].rename(columns={'fav_bsp': 'bsp', 'fav_won': 'won'}).copy()
h2_best['stake'] = FLAT_STAKE
if len(h2_best) >= 50:
    r_h2_best = backtest_back(h2_best)
    ax5.plot(r_h2_best['event_dt'], r_h2_best['bank'],
             color='purple', linewidth=1.5, label='Drift ≥10% + One-out')
ax5.axhline(STARTING_BANK, color='black', linewidth=0.8, linestyle='--')
ax5.set_title('H2: Morning Drift × One-Out Dominance')
ax5.set_ylabel('Bank ($)')
ax5.legend(fontsize=8)
ax5.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# Panel 6: H3 equity — lay contested
ax6 = axes[1, 2]
h3_best = _df[
    (_df['dominance'] < 1.10) & (_df['fav_bsp'].notna())
].rename(columns={'fav_bsp': 'bsp', 'fav_won': 'won'}).copy()
h3_best['liability'] = FLAT_STAKE
h3_best['stake']     = h3_best['liability'] / (h3_best['bsp'] - 1)
if len(h3_best) >= 50:
    r_h3_best = backtest_lay(h3_best)
    ax6.plot(r_h3_best['event_dt'], r_h3_best['bank'],
             color='darkred', linewidth=1.5, label='Lay fav (dominance <1.1×)')
ax6.axhline(STARTING_BANK, color='black', linewidth=0.8, linestyle='--')
ax6.set_title('H3: Lay Favourite in Contested Races')
ax6.set_ylabel('Bank ($)')
ax6.legend(fontsize=8)
ax6.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()
